# Exploración en la definición de modelos QCPINN

A continuación se realiza una corta exploracion por la aplicación de los modelos.

## 0. Importe de librerias

Importamos librerias propias de AWS y particulares de python

In [1]:
# Linea adicional para instalación de modulos pendientes
%pip install amazon-braket-pennylane-plugin torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Librerias de interés
import torch; import os; import sys
import time ; import logging

from torch             import nn
from torch.optim       import Adam
from torch.nn          import MSELoss
from datetime          import datetime
from math              import factorial
from matplotlib.ticker import FormatStrFormatter

import pennylane         as qml
import numpy             as np
import matplotlib.pyplot as plt

# Librerias propias de AWS
from braket.tracking       import Tracker
from braket.aws.aws_device import AwsDevice
from braket.circuits       import Circuit

## 1. Definición de parametros de simulador / QPU

Definimos backend a emplear para los cálculos

In [3]:
# Use Braket SDK Cost Tracking to estimate the cost to run this example
tracker = Tracker().start()

# Definición de dispositivo
ankaa = AwsDevice("arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3")
ankaa_emulator = ankaa.emulator()

In [3]:
# Use Braket SDK Cost Tracking to estimate the cost to run this example
tracker = Tracker().start()

# Definición de dispositivo
ankaa = AwsDevice("arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3")
ankaa_emulator = ankaa.emulator()

Veamos que tipo de celdas son soportadas en este dispositivo

In [5]:
ankaa.properties.paradigm.nativeGateSet

['rx', 'rz', 'iswap']

- Recordemos que para nuestro caso el circuito que estamos estudiando se basa en compuertas: RX, RY, CNOT

In [6]:
pi = torch.pi
verbatim_circuit = Circuit().add_verbatim_box(
    Circuit()
    .rz(1, pi/2)
    .iswap(0, 1)
    .iswap(1, 2)
)

In [7]:
print(verbatim_circuit)

T  : │        0        │     1      │    2    │    3    │       4       │
                                     ┌───────┐                           
q0 : ───StartVerbatim────────────────┤ ISWAP ├─────────────EndVerbatim───
              ║                      └───┬───┘                  ║        
              ║         ┌──────────┐ ┌───┴───┐ ┌───────┐        ║        
q1 : ─────────║─────────┤ Rz(1.57) ├─┤ ISWAP ├─┤ ISWAP ├────────║────────
              ║         └──────────┘ └───────┘ └───┬───┘        ║        
              ║                                ┌───┴───┐        ║        
q2 : ─────────╨────────────────────────────────┤ ISWAP ├────────╨────────
                                               └───────┘                 
T  : │        0        │     1      │    2    │    3    │       4       │


In [8]:
emulator_run = ankaa_emulator.run(verbatim_circuit, shots=1)

emulator_result = emulator_run.result()

emulator_counts = emulator_result.measurement_counts

emulator_counts

Counter({'000': 1})

Con esto en mente, veamos que debemos considerar el entrenamiento con circuitos cuanticos que permitan entrenar empleando esta celdas y generar simulaciones que permitan validar la viabilidad del circuito a desarollar

## 2. Definición del modelo hibdrido
Definimos los parametros requeridos para el modelo hibrido, adicional de las funcionnes necesarias para los procesos requeridos, inicialmente consideraremos la simulación de los disparos cuanticos

In [4]:
# ----------------------------------------- Logger definition ---------------------------------------- #

class Logging:
    """_summary_: My custom logger"""

    def __init__(self, log_path, experiment_name=None, source_file=None):
        """_summary_

        Args:
            log_path (_type_): The parent (checkpoint) directory
            log_file (_type_): The name of the log file
            experiment_name (_type_, optional): Experiment or mode name
        """
        self.log_path = log_path
        self.experiment_name = experiment_name

        self._create_output_dir(source_file)
        self._create_logger()

    def _create_output_dir(self, source_path=None):
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S-%f")
        if self.experiment_name is not None:
            timestamp = f"{timestamp}_{self.experiment_name}"

        try:
            self.output_dir = os.path.join(self.log_path, timestamp)
            os.makedirs(self.output_dir, exist_ok=True)
            ## to copy training file in the log directory
            # destination_path = os.path.join(self.output_dir, "training.py")
            # if source_path is not None:
            # shutil.copy(source_path, destination_path)

        except OSError as error:
            print(f"Error: {error.strerror}")

    def _create_logger(self):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.DEBUG)

        logging.basicConfig(filemode="w")

        # sh = logging.StreamHandler()
        # sh.setLevel(logging.DEBUG)
        # sh.setFormatter(logging.Formatter("%(message)s"))
        # self.logger.addHandler(sh)

        fh = logging.FileHandler(f"{self.output_dir}/output.log")

        self.logger.addHandler(fh)

    def get_output_dir(self):
        """_summary_: The output directory where the log (e.g., output.log) is created

        Returns:
            _type_: Returns the out put directory where the log file is created
        """
        return self.output_dir

    def print(self, *args):
        """_summary_:"""

        # TODO: Check this code later

        for arg in args:
            if len(args) == 1:
                self.logger.info(arg)
            elif arg != args[-1]:
                for handler in self.logger.handlers:
                    handler.terminator = ""
                if (
                    type(arg) == float
                    or type(arg) == np.float64
                    or type(arg) == np.float32
                ):
                    self.logger.info("%.4e" % (arg))
                else:
                    self.logger.info(arg)
            else:
                for handler in self.logger.handlers:
                    handler.terminator = "\n"
                if (
                    type(arg) == float
                    or type(arg) == np.float64
                    or type(arg) == np.float32
                ):
                    self.logger.info("%.4e" % (arg))
                else:
                    self.logger.info(arg)


In [5]:
# ---------------------------------------- Plotter definition ---------------------------------------- #
def plt_prediction(logger, X_star, u_star, u_pred, f_star, f_pred):
    # Set global font sizes
    plt.rcParams.update(
        {
            "font.size": 14,
            "axes.titlesize": 16,
            "axes.labelsize": 14,
            "xtick.labelsize": 12,
            "ytick.labelsize": 12,
        }
    )

    # Data dictionary for u(x) and f(x) to be used in loops
    data = {
        "u": {
            "exact": u_star,
            "predicted": u_pred,
            "error": np.abs(u_star - u_pred),
            "title": r"$u(x)$",
        },
        "f": {
            "exact": f_star,
            "predicted": f_pred,
            "error": np.abs(f_star - f_pred),
            "title": r"$f(x)$",
        },
    }

    fig, axs = plt.subplots(2, 3, figsize=(18, 10))  # 2 rows, 3 columns
    content = ["exact", "predicted", "error"]

    x_unique = np.unique(X_star[:, 0])
    y_unique = np.unique(X_star[:, 1])
    X, Y = np.meshgrid(x_unique, y_unique)

    for row, (key, value) in enumerate(data.items()):
        for col, field in enumerate(content):
            Z = value[field].reshape(len(y_unique), len(x_unique))

            contour = axs[row, col].contourf(
                X,
                Y,
                Z,
                levels=20,  # Number of contour levels
                cmap="coolwarm",
            )

            # Only show y-axis labels and ticks for leftmost column
            if (col == 0) and (
                row == (len(data.items()) - 1)
            ):  # If not leftmost column
                axs[row, col].set_ylabel(r"$x_2$ →", fontsize=14)
                axs[row, col].set_xlabel(r"$x_1$ →", fontsize=14)
            else:
                axs[row, col].set_yticklabels([])
                axs[row, col].set_xticklabels([])
                axs[row, col].set_ylabel("")
                axs[row, col].set_xlabel("")

            axs[row, col].set_title(
                f"{field.capitalize()} {value['title']}", fontsize=16
            )

            # Add colorbar with larger font size
            cbar = fig.colorbar(contour, ax=axs[row, col])
            cbar.ax.tick_params(labelsize=12)

    # Adjust layout and save the figure
    plt.tight_layout()
    path = os.path.join(logger.get_output_dir(), "prediction.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_contour(
    X_star, u_star, img_name, plot_xy=False, xy_labels=[r"$x_1$", r"$x_2$"]
):
    """ """
    fig, axs = plt.subplots(1, 1, figsize=(5, 4))

    min_val = np.min(u_star)
    max_val = np.max(u_star)
    if min_val == max_val == 0:
        min_val += -1e-16
        max_val += 1e-6

    levels = np.linspace(min_val, max_val, 60)

    contour = axs.contourf(
        X_star[..., 0],
        X_star[..., 1],
        u_star,
        levels=levels,
        cmap="jet",
        vmin=min_val,
        vmax=max_val,
    )

    if plot_xy:
        axs.set_xlabel(xy_labels[0], fontsize=18, color="grey")
        axs.set_ylabel(xy_labels[1], fontsize=18, color="grey")

    axs.tick_params(axis="both", which="major", labelsize=14, colors="grey")

    cbar = fig.colorbar(
        contour, ax=axs, ticks=np.linspace(min_val, max_val, 4), pad=0.01
    )
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    cbar.ax.tick_params(labelsize=14, colors="grey")
    cbar.outline.set_visible(False)

    axs.spines["top"].set_visible(False)
    axs.spines["right"].set_visible(False)
    axs.spines["left"].set_visible(False)
    axs.spines["bottom"].set_visible(False)

    # Adjust layout and save the figure with reduced gap and dark grey border
    # plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
    plt.savefig(img_name, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    plt.close()


def grid_one_contour_plots_regular(
    data,
    x,
    y,
    dirname,
    plot_xy=False,
    xy_labels=[r"$t$", r"$x$"],
    img_width=4,
    img_height=5,
    ticks=4,
    fontsize=3,
    labelsize=3,
):
    fig, ax = plt.subplots(figsize=(img_width, img_height))

    min_ = np.min(data)
    max_ = np.max(data)
    if min_ == max_ == 0:
        min_ += -1e-16
        max_ += 1e-6

    levels = np.linspace(min_, max_, 60)
    contour = ax.contourf(x, y, data, levels=levels, cmap="jet", vmin=min_, vmax=max_)

    cbar = fig.colorbar(
        contour, ax=ax, ticks=np.linspace(min_, max_, ticks), format="%.1e"
    )
    cbar.ax.tick_params(labelsize=labelsize)

    if plot_xy:
        ax.set_xlabel(xy_labels[0], fontsize=fontsize)
        ax.set_ylabel(xy_labels[1], fontsize=fontsize)

    ax.tick_params(labelsize=labelsize)
    plt.savefig(dirname, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return fig


In [6]:
# ---------------------------------------- DVSolver definition --------------------------------------- #
# ALR: Separate definition of complex mse loss
def complex_mse_loss_magnitude(input_complex, target_complex):
        # Calculate the complex difference
        complex_diff = input_complex - target_complex

        # Calculate the squared magnitude (real part of complex_diff * complex_diff.conj())
        # Or, more simply, (complex_diff.real**2 + complex_diff.imag**2).mean()
        loss = torch.mean(complex_diff.real**2 + complex_diff.imag**2)
        return loss

class DVPDESolver(nn.Module):
    def __init__(self, args, logger: Logging, data=None, device=None):
        super().__init__()
        self.logger = logger
        self.device = device
        self.args = args
        self.data = data
        self.batch_size = self.args["batch_size"]
        self.num_qubits = self.args["num_qubits"]
        self.epochs = self.args["epochs"]
        self.optimizer = None
        self.scheduler = None
        self.loss_history = []
        self.encoding = self.args.get("encoding", "angle")
        self.draw_quantum_circuit_flag = True
        self.classic_network = self.args["classic_network"]  # [3, 50, 50, 50, 4] #

        if self.encoding == "amplitude":
            self.preprocessor = nn.Sequential(
                nn.Linear(self.classic_network[0], self.classic_network[-2]).to(
                    self.device
                ),
                nn.Tanh(),
                nn.Linear(self.classic_network[-2], self.num_qubits).to(self.device),
            ).to(self.device)
        else:
            self.preprocessor = nn.Sequential(
                nn.Linear(self.classic_network[0], self.classic_network[-2]).to(
                    self.device
                ),
                nn.Tanh(),
                nn.Linear(self.classic_network[-2], self.num_qubits).to(self.device),
            ).to(self.device)

        self.postprocessor = nn.Sequential(
            nn.Linear(self.num_qubits, self.classic_network[-2]).to(self.device),
            nn.Tanh(),
            nn.Linear(self.classic_network[-2], self.classic_network[-1]).to(
                self.device
            ),
        ).to(self.device)

        self.activation = nn.Tanh()

        # Quantum parameters
        self.num_qubits = args["num_qubits"]
        self.quantum_layer = DVQuantumLayer(self.args)

        self.optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.parameters()), lr=self.args["lr"]
        )

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.9, patience=1000
        )

        # ALR: Loss function selection by args
        if self.args['problem'] == 'complex_wave':
            self.loss_fn = complex_mse_loss_magnitude
        else:
            self.loss_fn = torch.nn.MSELoss()

        self._initialize_logging()
        self._initialize_weights()

    def _initialize_weights(self):
        """Apply Xavier initialization to all layers."""
        for layer in self.preprocessor:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(
                    layer.weight
                )  # Or use xavier_normal_ for normal distribution
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)  # Set biases to zero

    def _initialize_logging(self):
        self.log_path = self.logger.get_output_dir()

        # self.logger.print(f"checkpoint path: {self.log_path=}")

        # # total number of parameters
        # total_params = sum(p.numel() for p in self.parameters())
        # print(f"Total number of parameters: {total_params}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the hybrid network
        Args:
            x: Spatial coordinates
            t: Time coordinates
        Returns:
            PDE solution values
        """

        try:
            if x.dim() != 2:
                raise ValueError(f"Expected 2D input tensor, got shape {x.shape}")
            # Combine inputs
            # Classical preprocessing
            preprocessed = self.preprocessor(x)
            # print(f"preprocessed: {preprocessed.shape}")
            # Quantum processing

            if self.draw_quantum_circuit_flag:
                self.draw_quantum_circuit(preprocessed)
                self.draw_quantum_circuit_flag = False

            quantum_out = self.quantum_layer(preprocessed).to(
                dtype=torch.float32, device=self.device
            )
            # print(f"quantum_out: {quantum_out.shape}")

            classical_out = self.postprocessor(quantum_out)
            # print(f"classical_out: {classical_out.shape}")
            return classical_out

        except Exception as e:
            self.logger.print(f"Forward pass failed: {str(e)}")
            raise

    # Save model state to a file
    def save_state(self):
        state = {
            "args": self.args,
            "classic_network": self.classic_network,
            "quantum_params": self.quantum_layer.state_dict(),
            "preprocessor": self.preprocessor.state_dict(),
            "quantum_layer": self.quantum_layer.state_dict(),
            "postprocessor": self.postprocessor.state_dict(),
            # "classical_input_scale": self.classical_input_scale.detach().cpu().numpy(),
            # "classical_output_scale": self.classical_output_scale.detach().cpu().numpy(),
            "optimizer": self.optimizer.state_dict(),
            "scheduler": self.scheduler.state_dict(),
            "loss_history": self.loss_history,
            "log_path": self.log_path,
        }

        model_path_state   = os.path.join(self.log_path, "model.pth")
        model_path_weights = os.path.join(self.log_path, "model_weights.pth")

        with open(model_path_state, "wb") as f:
            torch.save(state, f)

        self.logger.print(f"Model state saved to {model_path_state}")

        # Additional line to save current state
        torch.save(self.state_dict(), model_path_weights)

        self.logger.print(f"Model state saved to {model_path_weights}")

    # Load model state from a file
    @classmethod
    def load_state(cls, file_path, map_location=None):
        if map_location is None:
            map_location = torch.device("cpu")
        with open(file_path, "rb") as f:
            state = torch.load(f, map_location=map_location)
            # state = pickle.load(f)
        print(f"Model state loaded from {file_path}")
        return state

    def draw_quantum_circuit(self, x):
        if self.draw_quantum_circuit_flag:
            try:
                self.logger.print("The circuit used in the study:")
                if self.quantum_layer.params is not None:
                    fig, ax = qml.draw_mpl(self.quantum_layer.circuit)(x[0])
                    plt.savefig(os.path.join(self.log_path, "circuit.pdf"))
                    plt.close()
                    print(f"The circuit is saved in {self.log_path}")
            except Exception as e:
                self.logger.print(f"Failed to draw quantum circuit: {str(e)}")


In [7]:
# ---------------------------------------- Samplers definition --------------------------------------- # 

# Sample collocation of points
def sample_collocation(Nf, Nb, N0, L=1.0, T=0.2, device="cpu", dtype=torch.float64, example= 'box'):
    
    # Harmonic portential points
    if example.lower() == 'ho':
        # Interior (PDE)
        t_f = torch.rand(Nf,1, device=device, dtype=dtype)*T
        x_f = (torch.rand(Nf,1, device=device, dtype=dtype)-0.5)*2*L
        # Bordes (x=0 y x=L)
        t_b = torch.rand(Nb, 1, device=device, dtype=dtype) * T
        xb0 = torch.full(     (Nb//2, 1), -L, device=device, dtype=dtype)
        xbL = torch.full((Nb - Nb//2, 1),  L, device=device, dtype=dtype)
        x_b = torch.cat([xb0, xbL], dim=0)
        t_b = torch.cat([t_b[:Nb//2], t_b[Nb//2:]], dim=0)
        # Inicial (t=0)
        t_0 = torch.zeros(N0, 1, device=device, dtype=dtype)
        x_0 = (torch.rand(N0,1, device=device)-0.5)*2*L

    # Box potential points
    else:
        # Interior (PDE)
        t_f = torch.rand(Nf, 1, device=device, dtype=dtype) * T
        x_f = torch.rand(Nf, 1, device=device, dtype=dtype) * L
        # Bordes (x=0 y x=L)
        t_b = torch.rand(Nb, 1, device=device, dtype=dtype) * T
        xb0 = torch.zeros(Nb//2, 1, device=device, dtype=dtype)
        xbL = torch.full((Nb - Nb//2, 1), L, device=device, dtype=dtype)
        x_b = torch.cat([xb0, xbL], dim=0)
        t_b = torch.cat([t_b[:Nb//2], t_b[Nb//2:]], dim=0)
        # Inicial (t=0)
        t_0 = torch.zeros(N0, 1, device=device, dtype=dtype)
        x_0 = torch.rand(N0, 1, device=device, dtype=dtype) * L


    return (t_f, x_f), (t_b, x_b), (t_0, x_0)

# Definition of exact eigen state of analysis
def exact_eigenstate(n, t, x, L=1.0, mass=1.0, hbar=1.0, omega= 1.0, example= 'box'):

    # Harmonic Oscillator solution
    if example.lower() == 'ho':

        # Hermite polynomials definition
        def hermite_physicists(n, z):
            if n == 0: return torch.ones_like(z)
            if n == 1: return 2.0*z
            Hnm2 = torch.ones_like(z); Hnm1 = 2.0*z
            for k in range(1, n):
                Hn = 2.0*z*Hnm1 - 2.0*k*Hnm2
                Hnm2, Hnm1 = Hnm1, Hn
            return Hnm1

        xi   = np.sqrt(mass*omega/hbar) * x
        fact = factorial(n)
        norm = (mass*omega/(np.pi*hbar))**0.25 * (1.0/np.sqrt((2.0**n)*fact))
        phi_x = norm * hermite_physicists(n, xi) * torch.exp(-0.5*(xi**2))
        E_n   = hbar*omega*(n + 0.5)
        phase = -(E_n/hbar) * t
        psi_r = phi_x * torch.cos(phase)
        psi_i = phi_x * torch.sin(phase)

    # Box potential solution
    else:
        pi = torch.tensor(np.pi, device=t.device, dtype=t.dtype)
        k = n * pi / L
        E_n = (n**2) * (pi**2) * (hbar**2) / (2.0 * mass * (L**2))
        phase = - (E_n / hbar) * t
        spatial = torch.sqrt(2.0 / torch.tensor(L, device=t.device, dtype=t.dtype)) * torch.sin(k * x)
        psi_r = spatial * torch.cos(phase)
        psi_i = spatial * torch.sin(phase)

    return psi_r, psi_i, E_n

In [8]:
# ---------------------------------------- Operator definition --------------------------------------- # 

# ALR: Additional operator separated -> Schrödinger direct
def schrodinger_operator(model, t, x, potential_fn=0, mass=1.0, hbar=1.0):
    """
    Residuos de: i*hbar*psi_t = -(hbar^2/(2m)) * psi_xx + V * psi
    model(t,x) -> [psi_r, psi_i]
    """

    t = t.requires_grad_(True)
    x = x.requires_grad_(True)

    psi = model(torch.concatenate((t, x), dim=1))
    psi_r = psi[:, 0:1]
    psi_i = psi[:, 1:2]

    # Derivadas temporales
    psi_t_r = torch.autograd.grad(psi_r, t, torch.ones_like(psi_r), create_graph=True)[0]
    psi_t_i = torch.autograd.grad(psi_i, t, torch.ones_like(psi_i), create_graph=True)[0]

    # Derivadas espaciales segunda
    psi_x_r  = torch.autograd.grad(psi_r, x, torch.ones_like(psi_r), create_graph=True)[0]
    psi_x_i  = torch.autograd.grad(psi_i, x, torch.ones_like(psi_i), create_graph=True)[0]
    psi_xx_r = torch.autograd.grad(psi_x_r, x, torch.ones_like(psi_x_r), create_graph=True)[0]
    psi_xx_i = torch.autograd.grad(psi_x_i, x, torch.ones_like(psi_x_i), create_graph=True)[0] 

    V = potential_fn(t, x) if potential_fn!=0 else torch.zeros_like(psi_r)
    coef = (hbar**2) / (2.0 * mass)

    residual_r = -hbar * psi_t_i + coef * psi_xx_r - V * psi_r
    residual_i =  hbar * psi_t_r + coef * psi_xx_i - V * psi_i

    return psi_r, psi_i, residual_r, residual_i

In [33]:
# -------------------------------------- Quantum Layer definition ------------------------------------ #
class DVQuantumLayer(nn.Module):
    def __init__(self, args, diff_method="best"):
        super().__init__()

        """
        Initialize the quantum layer with the given number of qubits and arguments.

        Args:
            num_qubits (int): Number of qubits in the quantum circuit.
            args (dict): Additional arguments for the quantum circuit (e.g., hyperparameters).
            diff_method (str): Differentiation method for the QNode (default: "backprop").
        """
        self.num_qubits = args["num_qubits"]
        self.num_quantum_layers = args["num_quantum_layers"]
        self.shots = args["shots"]
        self.q_ansatz = args["q_ansatz"]
        self.problem = args["problem"]
        self.encoding = args.get("encoding", "angle")

        # Variable por shot counting
        self.shots_done = 0

        if self.q_ansatz == "layered_circuit":
            self.params = nn.Parameter(
                torch.empty(
                    self.num_quantum_layers,
                    self.num_qubits * 4,
                    requires_grad=True,
                    dtype=torch.float32,
                )
            )

        elif self.q_ansatz == "alternating_layer_tdcnot":
            self.params = nn.Parameter(
                torch.empty(
                    self.num_quantum_layers,
                    self.num_qubits * 4,
                    requires_grad=True,
                    dtype=torch.float32,
                )
            )
        elif self.q_ansatz == "sim_circ_19":
            self.params = nn.Parameter(
                torch.empty(
                    self.num_quantum_layers,
                    self.num_qubits * 3,
                    requires_grad=True,
                    dtype=torch.float32,
                )
            )

        elif self.q_ansatz == "farhi":
            self.params = nn.Parameter(
                torch.empty(
                    self.num_quantum_layers,
                    (2 * self.num_qubits - 2),
                    requires_grad=True,
                    dtype=torch.float32,
                )
            )

        elif self.q_ansatz == "sim_circ_15":
            self.params = nn.Parameter(
                torch.empty(
                    self.num_quantum_layers,
                    self.num_qubits * 2,
                    requires_grad=True,
                    dtype=torch.float32,
                )
            )

        elif self.q_ansatz == "sim_circ_5":
            self.params = nn.Parameter(
                torch.empty(
                    self.num_quantum_layers,
                    (3 * self.num_qubits) * self.num_qubits,
                    requires_grad=True,
                    dtype=torch.float32,
                )
            )
        else:
            self.params = None

        if not hasattr(self, "params") or self.params is None:
            raise ValueError(
                "Parameters are not initialized. Check the q_ansatz value."
            )
        self._initialize_weights()

        # ALR: Try with other devices, like emulators
        # -> arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3
        # -> arn:aws:braket:::device/quantum-simulator/amazon/sv1
        self.dev = qml.device("braket.aws.qubit", wires=self.num_qubits, shots=self.shots,
                             device_arn="arn:aws:braket:::device/quantum-simulator/amazon/sv1")
        self.circuit = qml.QNode(
            self._quantum_circuit, self.dev, interface="torch", diff_method=diff_method
        )

    def _quantum_circuit(self, x):
        if self.encoding == "amplitude":
            qml.templates.AmplitudeEmbedding(
                x, wires=range(self.num_qubits), normalize=True, pad_with=0.0
            )
        else:
            qml.templates.AngleEmbedding(x, wires=range(self.num_qubits), rotation="X")

        if self.q_ansatz == "layered_circuit":
            for layer in range(self.num_quantum_layers):
                self.layered_circuit(self.params[layer])

        elif self.q_ansatz == "alternating_layer_tdcnot":
            for layer in range(self.num_quantum_layers):
                self.alternating_layer_tdcnot(self.params[layer])
        elif self.q_ansatz == "sim_circ_19":
            for layer in range(self.num_quantum_layers):
                self.sim_circ_19(self.params[layer])
        elif self.q_ansatz == "farhi":
            for layer in range(self.num_quantum_layers):
                self.farhi_ansatz(self.params[layer])

        elif self.q_ansatz == "sim_circ_15":
            for layer in range(self.num_quantum_layers):
                self.create_sim_circuit_15(self.params[layer])

        elif self.q_ansatz == "sim_circ_5":
            for layer in range(self.num_quantum_layers):
                self.create_circuit_5(self.params[layer])

        return [qml.expval(qml.PauliZ(i)) for i in range(self.num_qubits)]

    def _initialize_weights(self):
        """Apply Xavier initialization to all layers."""

        if self.q_ansatz == "farhi":
            torch.nn.init.xavier_normal_(
                self.params.view(self.num_quantum_layers, (2 * self.num_qubits - 2))
            )
        elif self.q_ansatz in ["sim_circ_15"]:
            torch.nn.init.xavier_normal_(
                self.params.view(self.num_quantum_layers, self.num_qubits * 2)
            )
        elif self.q_ansatz in ["layered_circuit", "alternating_layer_tdcnot"]:
            torch.nn.init.xavier_normal_(
                self.params.view(self.num_quantum_layers, self.num_qubits * 4)
            )
        elif self.q_ansatz == "sim_circ_19":
            torch.nn.init.xavier_normal_(
                self.params.view(self.num_quantum_layers, self.num_qubits * 3)
            )
        elif self.q_ansatz == "sim_circ_5":
            torch.nn.init.xavier_normal_(
                self.params.view(
                    self.num_quantum_layers, (3 * self.num_qubits) * self.num_qubits
                )
            )
        else:
            raise ValueError("Invalid q_ansatz value.", self.q_ansatz)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # return torch.stack([self.circuit(sample) for sample in x])

        # ALR: Compatibility modification
        self.shots_done += x.shape[0]

        return torch.stack([torch.hstack(self.circuit(sample)) for sample in x])

    def layered_circuit(self, params):
        """
        Creates a quantum circuit with num_layers * num_qubits parameters.

        Args:
            params (list or tensor): A flat list or tensor of parameters with length num_layers * num_qubits.
            num_qubits (int): The number of qubits in the circuit.
            num_layers (int): The number of layers in the circuit.

        Returns:
            None: Constructs the quantum circuit.
        """
        assert params is not None and len(params) == self.num_qubits * 4, (
            "The number of parameters must be equal to 4* num_qubits."
        )

        # track the parameter index
        param_idx = 0

        # print(f"{len(params)=}")

        # apply RZ and RX gates for each qubit in the layer
        for qubit_id in range(self.num_qubits):
            # print(f"layer: {layer} ,{param_idx=}")
            qml.RZ(params[param_idx], wires=qubit_id)
            param_idx += 1
            qml.RX(params[param_idx], wires=qubit_id)
            param_idx += 1

        qml.Barrier(wires=range(self.num_qubits))

        for qubit_id in range(self.num_qubits):
            qml.CNOT(wires=[qubit_id, (qubit_id + 1) % self.num_qubits])

        qml.Barrier(wires=range(self.num_qubits))

        for qubit_id in range(self.num_qubits):
            # print(f"layer: {layer} ,{param_idx=}")
            qml.RX(params[param_idx], wires=qubit_id)
            param_idx += 1
            qml.RZ(params[param_idx], wires=qubit_id)
            param_idx += 1

    def alternating_layer_tdcnot(self, params):
        """
        Build a variational circuit with alternating thinly dressed CNOT gates.

        Args:
            params (list or np.ndarray): Parameters for the circuit. Should have a size of
                                        `num_layers * num_qubits * 4` (4 parameters per thinly dressed CNOT gate).
        """
        assert params is not None and len(params) == self.num_qubits * 4, (
            "The number of parameters must be equal to  num_qubits * 4."
        )

        param_idx = 0  # Initialize the parameter index

        def build_tdcnot(ctrl, tgt):
            """Build a thinly dressed CNOT gate with the required parameters."""
            nonlocal param_idx  # Allow modification of the outer variable
            qml.RY(params[param_idx], wires=ctrl)
            param_idx += 1
            qml.RY(params[param_idx], wires=tgt)
            param_idx += 1
            qml.CNOT(wires=[ctrl, tgt])
            qml.RZ(params[param_idx], wires=ctrl)
            param_idx += 1
            qml.RZ(params[param_idx], wires=tgt)
            param_idx += 1

        # add layers of the ansatz
        for i in range(self.num_qubits - 1)[::2]:
            ctrl, tgt = i, ((i + 1) % self.num_qubits)
            build_tdcnot(ctrl, tgt)

        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))

        for i in range(self.num_qubits)[1::2]:
            ctrl, tgt = i, ((i + 1) % self.num_qubits)
            build_tdcnot(ctrl, tgt)

    def sim_circ_19(self, params):
        def add_rotations():
            param_counter = 0
            for i in range(0, self.num_qubits):
                qml.RX(params[param_counter], wires=i)
                param_counter += 1

            for i in range(0, self.num_qubits):
                qml.RZ(params[param_counter], wires=i)
                param_counter += 1

            # barrier after entanglement
            qml.Barrier(wires=range(self.num_qubits))

        def add_entangling_gates():
            param_counter = 0
            qml.CRX(params[param_counter], wires=[self.num_qubits - 1, 0])
            param_counter += 1
            for i in reversed(range(1, self.num_qubits)):
                qml.CRX(params[param_counter], wires=[i - 1, i])
                param_counter += 1

        # add layers of the ansatz
        add_rotations()
        add_entangling_gates()

    def farhi_ansatz(self, params):
        param_counter = 0

        # ensure there are enough parameters for both sets of gates
        if len(params) != (2 * self.num_qubits - 2):
            raise ValueError("Insufficient parameters for RXX and RZX gates")

        # custom RXX and RZX gate definitions
        def RXX(theta, wires):
            qml.CNOT(wires=wires)
            qml.RX(theta, wires=wires[0])
            qml.CNOT(wires=wires)

        def RZX(theta, wires):
            qml.CNOT(wires=wires)
            qml.RZ(theta, wires=wires[0])
            qml.CNOT(wires=wires)

        # RXX gates
        for i in range(self.num_qubits - 1):
            RXX(params[param_counter], wires=[self.num_qubits - 1, i])
            param_counter += 1

        # RZX gates
        for i in range(self.num_qubits - 1):
            RZX(params[param_counter], wires=[self.num_qubits - 1, i])
            param_counter += 1

    def create_sim_circuit_15(self, params):
        """
        Creates a variational circuit based on circuit 15 in arXiv:1905.10876.

        Args:
            n_data_qubits (int): Number of qubits in the circuit
            layers (int): Number of layers in the circuit
            sweeps_per_layer (int): Number of sweeps per layer
            activation_function (callable, optional): Activation function to apply between layers

        Returns:
            callable: A function that constructs the quantum circuit with given parameters
        """
        if params is None or len(params) != 2 * self.num_qubits:
            raise ValueError("Insufficient parameters for RXX and RZX gates")

        param_index = 0
        
        # ALR: Final modification, gate manipulable with simlators on AWS 
        # Originals: 
        # - apply_rotations1 -> RY  
        # - apply_rotations2 -> Rx  
        # - apply_entangling_block1 -> CNOT 
        # - apply_entangling_block2 -> CNOT 
        
        # apply rotations
        def apply_rotations1():
            nonlocal param_index
            for i in range(self.num_qubits):
                qml.RZ(params[param_index], wires=i)
                param_index += 1

        def apply_rotations2():
            nonlocal param_index
            for i in range(self.num_qubits):
                qml.RZ(params[param_index], wires=i)
                param_index += 1

        # apply entangling gates block 1
        def apply_entangling_block1():
            for i in reversed(range(self.num_qubits)):
                qml.ISWAP(wires=[i, (i + 1) % self.num_qubits])

        # apply entangling gates block 2
        def apply_entangling_block2():
            for i in range(self.num_qubits):
                control_qubit = (i + self.num_qubits - 1) % self.num_qubits
                target_qubit = (control_qubit + 3) % self.num_qubits
                qml.ISWAP(wires=[control_qubit, target_qubit])

        # main circuit construction
        apply_rotations1()
        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))
        apply_entangling_block1()
        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))
        apply_rotations2()
        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))
        apply_entangling_block2()
        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))

    def create_circuit_5(self, params):
        """
        Creates a generalized version of Circuit 5 with CRZ gates where control and target wires
        are properly separated.

        Args:
            params (np.ndarray): Array of parameters for the rotation gates
        """
        param_idx = 0
        # verify parameter count
        expected_params = (3 * self.num_qubits) * self.num_qubits

        if params is None or len(params) != expected_params:
            raise ValueError(
                f"Expected {expected_params} parameters but got {params.shape}"
            )

        # initial Rx gates on all qubits
        for i in range(self.num_qubits):
            qml.RX(params[param_idx], wires=i)
            param_idx += 1

        for i in range(self.num_qubits):
            qml.RZ(params[param_idx], wires=i)
            param_idx += 1

        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))
        # additional Rz gates for all except last qubit
        for i in range(self.num_qubits - 1, -1, -1):
            for j in range(self.num_qubits - 1, -1, -1):
                if j != i:
                    qml.CRZ(params[param_idx], wires=[i, j])
                    param_idx += 1

        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))

        # middle layer (using RX instead of CNOTs as in original)
        for i in range(self.num_qubits):
            qml.RX(params[param_idx], wires=i)
            param_idx += 1

        for i in range(self.num_qubits):
            qml.RZ(params[param_idx], wires=i)
            param_idx += 1

        # barrier after entanglement
        qml.Barrier(wires=range(self.num_qubits))

    def quantum_tanh_n_qubits(self, params, scale=1.0):
        """
        Enhanced nonlinear quantum tanh activation with cross-qubit interactions

        Args:
            scale (float): Scaling factor for the activation
            params (list): List of trainable parameters for the rotations
        """
        if self.num_qubits is None:
            raise ValueError("Wires cannot be None.")

        if params is None:
            # create parameters for both direct and cross interactions
            n_params = self.num_qubits * (self.num_qubits - 1) // 2
            params = [scale * np.pi / 2.0 * index for index in range(n_params)]

        # add nonlinear phase shifts
        for index in range(self.num_qubits):
            qml.PhaseShift(np.sin(params[index]) * np.pi, wires=index)


In [34]:
# ------------------------------ Neural Network Architecture definition ------------------------------ #
class CVNeuralNetwork2(nn.Module):
    """
    Implementation of CV Neural Network based on https://arxiv.org/pdf/1806.06871
    Following equation 26 structure: Linear -> Non-linear -> Linear
    """

    def __init__(
        self,
        num_qumodes: int,
        num_layers: int,
        device: str = "cpu",
        cutoff_dim: int = 2,
    ):
        super().__init__()
        self.num_qumodes = num_qumodes
        self.num_layers = num_layers
        self.cutoff_dim = cutoff_dim
        self.device = device
        active_sd = 0.1
        passive_sd = 2 * np.pi
        # Initialize trainable parameters
        # self.weights = self._initialize_weights()

        # Initialize trainable parameters
        # Parameters for interferometers (linear transformations)

        self.num_interfermoter_params = int(
            self.num_qumodes * (self.num_qumodes - 1)
        ) + max(1, self.num_qumodes - 1)

        self.theta_1 = nn.Parameter(
            torch.randn(num_layers, self.num_interfermoter_params, device=self.device)
            * passive_sd,
            requires_grad=True,
        )

        self.theta_2 = nn.Parameter(
            torch.randn(num_layers, self.num_interfermoter_params, device=self.device)
            * passive_sd,
            requires_grad=True,
        )
        self.squeezing_r = nn.Parameter(
            torch.randn(num_layers, num_qumodes, device=self.device) * active_sd,
            requires_grad=True,
        )
        self.squeezing_phi = nn.Parameter(
            torch.randn(num_layers, num_qumodes, device=self.device) * passive_sd,
            requires_grad=True,
        )
        # Parameters for non-linear transformations
        self.displacement_r = nn.Parameter(
            torch.randn(num_layers, num_qumodes, device=self.device) * active_sd,
            requires_grad=True,
        )
        self.displacement_phi = nn.Parameter(
            torch.randn(num_layers, num_qumodes, device=self.device) * passive_sd,
            requires_grad=True,
        )
        # Add Kerr parameters
        self.kerr_params = nn.Parameter(
            torch.randn(num_layers, num_qumodes, device=self.device) * active_sd,
            requires_grad=True,
        )
        # Create quantum device: 
        # - arn:aws:braket:::device/quantum-simulator/amazon/sv1
        # - arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3
        self.dev = qml.device(
            "braket.aws.qubit", wires=num_qumodes, cutoff_dim=cutoff_dim,
            shots=self.shots,
            device_arn="arn:aws:braket:::device/quantum-simulator/amazon/sv1"
        )

        # Create quantum node
        self.circuit = qml.QNode(self._quantum_circuit, self.dev, interface="torch")

        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the quantum neural network
        """

        # Classical forward pass, for instance we need the emulator one
        return torch.stack([self.circuit(sample) for sample in x])

    def _quantum_circuit(self, inputs):
        # Encode input x into quantum state
        # Encode inputs
        for i, input_val in enumerate(inputs):
            # print(f"input_val: {input_val}")
            qml.Displacement(input_val, 0.0, wires=i)

        # iterative quantum layers
        for layer_idx in range(self.num_layers):
            self.qnn_layer(layer_idx)

        return [
            qml.expval(qml.QuadOperator(wires=wire, phi=0.0))
            for wire in range(self.num_qumodes)
        ]

    def qnn_layer(self, layer_idx):
        """CV quantum neural network layer acting on ``N`` modes.

        Args:
            params (list[float]): list of length ``2*(max(1, N-1) + N**2 + n)`` containing
                the number of parameters for the layer
            q (list[RegRef]): list of Strawberry Fields quantum registers the layer
                is to be applied to
        """

        # qumode_list = list(range(self.num_qumodes))

        self.interferometer(self.theta_1[layer_idx])

        for wire in range(self.num_qumodes):
            qml.Squeezing(
                self.squeezing_r[layer_idx, wire],
                self.squeezing_phi[layer_idx, wire],
                wires=wire,
            )
            # qml.Squeezing(
            #     self.squeezing_r[layer_idx, wire],
            #     0.0,
            #     wires=wire,
            # )
            # ops.Sgate(s[i]) | q[i]

        self.interferometer(self.theta_2[layer_idx])

        for wire in range(self.num_qumodes):
            qml.Displacement(
                self.displacement_r[layer_idx, wire],
                self.displacement_phi[layer_idx, wire],
                wires=wire,
            )
            qml.Kerr(self.kerr_params[layer_idx, wire], wires=wire)

    def interferometer(self, params):
        """Parameterised interferometer acting on ``N`` modes.

        Args:
            params (list[float]): list of length ``max(1, N-1) + (N-1)*N`` parameters.

                * The first ``N(N-1)/2`` parameters correspond to the beamsplitter angles
                * The second ``N(N-1)/2`` parameters correspond to the beamsplitter phases
                * The final ``N-1`` parameters correspond to local rotation on the first N-1 modes

            q (list[RegRef]): list of Strawberry Fields quantum registers the interferometer
                is to be applied to
        """

        qumode_list = list(range(self.num_qumodes))

        theta = params[: self.num_qumodes * (self.num_qumodes - 1) // 2]
        phi = params[
            (self.num_qumodes * (self.num_qumodes - 1) // 2) : (
                self.num_qumodes * (self.num_qumodes - 1)
            )
        ]
        rphi = params[-self.num_qumodes + 1 :]

        if self.num_qumodes == 1:
            # the interferometer is a single rotation
            qml.Rotation(rphi[0], wires=0)
            return

        n = 0  # keep track of free parameters

        # Apply the rectangular beamsplitter array
        # The array depth is N
        for l in range(self.num_qumodes):
            for k, (q1, q2) in enumerate(zip(qumode_list[:-1], qumode_list[1:])):
                # skip even or odd pairs depending on layer
                if (l + k) % 2 != 1:
                    qml.Beamsplitter(theta[n], phi[n], wires=[q1, q2])
                    n += 1

        # apply the final local phase shifts to all modes except the last one
        for i in range(max(1, self.num_qumodes - 1)):
            qml.Rotation(rphi[i], qumode_list[i])


In [11]:
# ----------------------------------------  Trainer definition --------------------------------------- # 

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32  # mejor precisión para EDP de 2º orden

def train(model, N_f = 7, N_b = 5, N_0 = 5):

    # Parametros de decisión 
    example = model.args['eq_params']['example']    # Ejemplo a replicar

    # Parametros físicos del dominio
    L       = model.args['eq_params']['L']          # dominio espacial [0, L]
    T       = model.args['eq_params']['T']          # tiempo final
    hbar    = model.args['eq_params']['hbar'] 
    mass    = model.args['eq_params']['mass']
    n_level = model.args['eq_params']['n_level']    # nivel del pozo (usaremos n=1)

    # Definición por defecto de parametros
    potential_fn = 0; omega = 0

    # Cargado según ejemplos de uso
    if example.lower() == 'ho':                                        # Oscilador armonico

        potential_fn = eval(model.args['eq_params']['potential_fn'])   # Función de potencial
        omega        = model.args['eq_params']['omega']                # Frecuencia natural


    # Constants parameters
    LR = model.args['lr']; PRINT_EVERY = model.args['print_every']

    # Parameters definition by model
    opt = model.optimizer if model.optimizer is not None else Adam(model.parameters(), lr=LR)
    mse = model.loss_fn if model.loss_fn is not None else MSELoss() 
    
    DEVICE = model.args['device']; DTYPE = torch.float32 

    # Dtype definition 
    torch.set_default_dtype(DTYPE)
    torch.manual_seed(42);

    t0 = time.time()
 
    for epoch in range(1, model.epochs + 1): 

        # Points definition
        (t_f, x_f), (t_b, x_b), (t_0, x_0) = sample_collocation(N_f, N_b, N_0, L=L, T=T, device=DEVICE, dtype=DTYPE, example=example)
        psi0_r, psi0_i, _ = exact_eigenstate(n_level, t_0, x_0, L=L, mass=mass, hbar=hbar, omega=omega, example=example)

        # Preparation per epoch 
        opt.zero_grad()

        # PDE (interior) con V=0 (pozo interior)
        [_, _, rR, rI] = schrodinger_operator(model, t_f, x_f, potential_fn= potential_fn, mass=mass, hbar=hbar)
        loss_pde = mse(rR[:, 0:1], torch.zeros_like(rR)) + mse(rI[:, 0:1], torch.zeros_like(rI))

        # BC Dirichlet: ψ=0 en x=0 y x=L
        psi_b = model(torch.cat((t_b, x_b), dim=1))
        loss_bc = mse(psi_b[:, 0:1], torch.zeros_like(psi_b[:, 0:1])) + \
                  mse(psi_b[:, 1:2], torch.zeros_like(psi_b[:, 1:2]))

        # IC: ψ(t=0,x) = sqrt(2/L) sin(pi x/L) (parte imag=0 al inicio)
        psi0 = model(torch.cat((t_0, x_0), dim=1))
        loss_ic = mse(psi0[:, 0:1], psi0_r[:, 0:1]) + mse(psi0[:, 1:2], psi0_i[:, 0:1])

        # Ponderación básica 
        loss = 2.0 * loss_pde + 1.0 * loss_bc + 2.0 * loss_ic

        loss.backward()
        opt.step()

        if epoch % PRINT_EVERY == 0 or epoch == 1:
            elapsed = time.time() - t0 
            model.logger.print(
                    "It: %d, Loss: %.3e, Loss_res: %.3e,  Loss_bcs: %.3e, Loss_ut_ics: %.3e, lr: %.3e, Time: %.2e"
                    % (
                        epoch,
                        loss.item(),
                        loss_pde.item(),
                        loss_bc.item(),
                        loss_ic.item(),
                        model.optimizer.param_groups[0]["lr"] if model.optimizer else 0.0,
                        elapsed,
                )
            )

            # Compute and Print adaptive weights during training
            # Compute the adaptive constant
            model.save_state()

        # Save of loss for each epoch
        model.loss_history.append(loss.item())

Una vez con todos los elementos definidos, podemos así considerar la evaluación empleando circuitos compatibles, para esto consideraremos el circuito alterado construido y las definiciones a continuación

In [35]:
mode = "hybrid"
num_qubits = 5
output_dim = 2
input_dim = 2
hidden_dim = 25
num_quantum_layers = 1
cutoff_dim = 20
classic_network = [input_dim, hidden_dim, output_dim]

# Input parameters: Iterable
L = 5.0        # dominio espacial [0, L]
T = 0.1        # tiempo final
hbar = 1.0
mass = 1.0
n_level = 0    # nivel (usaremos n=1)
omega  = 1.0
example = 'ho'
potential_fn = 0

if example.lower() != 'box':
    potential_fn = f"lambda t, x : 0.5 * {mass} * ({omega}**2) * (x**2)"

# Equation parameters
eq_params = {
    'example' : example,             # Ejemplo a correr: "BOX", "HO" 
    'L': L,                          # dominio espacial [0, L]
    'T' : T,                         # tiempo final
    'hbar' : hbar,                   # Atomic coordinates = 1
    'mass' : mass,                   # Atomic coordinates = 1
    'n_level' : n_level,             # nivel del pozo (usaremos n=1)
    'omega'   : omega,               # Natural frequency
    'potential_fn' : potential_fn,   # potential function
}

args = {
    "batch_size": 64,
    "epochs": 1000,
    "lr": 1E-3,
    "seed": 42,
    "print_every": 10,
    "log_path": "./results/qho",
    "input_dim": input_dim,
    "output_dim": output_dim,
    "num_qubits": num_qubits,
    "hidden_dim": hidden_dim,
    "num_quantum_layers": num_quantum_layers,
    "classic_network": classic_network,
    "q_ansatz": "sim_circ_15",  # options: "alternating_layer_tdcnot", "abbas" , farhi , sim_circ_13_half, sim_circ_13 , sim_circ_14_half, sim_circ_14 , sim_circ_15 ,sim_circ_19
    "mode": mode,
    "activation": "null",  # options: "null", "partial_measurement_half" , partial_measurement_x, tanh (Classical)
    "shots": 10,  # Analytical gradients enabled
    "problem": "schrodinger",
    "solver": "DV",  # options : "CV", "Classical", "DV"
    "device": DEVICE,
    "method": "None",
    "cutoff_dim": cutoff_dim,  # num_qubits >= cutoff_dim
    "class": "CVNeuralNetwork2",  # options CVNeuralNetwork1, CVNeuralNetwork2, CVNeuralNetwork3
    "encoding": "None",  # options : "ampiltude" , "angle" for DV , none for others
    "eq_params": eq_params,
}

In [36]:
log_path = args["log_path"]
logger = Logging(log_path)

In [37]:
if args["solver"] == "CV":
    model = CVPDESolver(args, logger, DEVICE)
    model.logger.print("Using CV Solver")
elif args["solver"] == "Classical":
    model = ClassicalSolver2(args, logger, DEVICE)
    model.logger.print("Using Classical Solver")
else:
    model = DVPDESolver(args, logger, DEVICE)
    model.logger.print("Using DV Solver")

model.logger.print(f"The settings used:")
for key, value in args.items():
    model.logger.print(f"{key} : {value}")

INFO:__main__:Using DV Solver
INFO:__main__:The settings used:
INFO:__main__:batch_size : 64
INFO:__main__:epochs : 1000
INFO:__main__:lr : 0.001
INFO:__main__:seed : 42
INFO:__main__:print_every : 10
INFO:__main__:log_path : ./results/qho
INFO:__main__:input_dim : 2
INFO:__main__:output_dim : 2
INFO:__main__:num_qubits : 5
INFO:__main__:hidden_dim : 25
INFO:__main__:num_quantum_layers : 1
INFO:__main__:classic_network : [2, 25, 2]
INFO:__main__:q_ansatz : sim_circ_15
INFO:__main__:mode : hybrid
INFO:__main__:activation : null
INFO:__main__:shots : 10
INFO:__main__:problem : schrodinger
INFO:__main__:solver : DV
INFO:__main__:device : cpu
INFO:__main__:method : None
INFO:__main__:cutoff_dim : 20
INFO:__main__:class : CVNeuralNetwork2
INFO:__main__:encoding : None
INFO:__main__:eq_params : {'example': 'ho', 'L': 5.0, 'T': 0.1, 'hbar': 1.0, 'mass': 1.0, 'n_level': 0, 'omega': 1.0, 'potential_fn': 'lambda t, x : 0.5 * 1.0 * (1.0**2) * (x**2)'}


In [38]:
# Definición de tipado para entrenamiento
DTYPE = torch.float32
torch.set_default_dtype(DTYPE)
torch.manual_seed(69420)

# Print total number of parameters
total_params = sum(p.numel() for p in model.parameters())
model.logger.print(f"Total number of parameters: {total_params}")

INFO:__main__:Total number of parameters: 417


In [ ]:
train(model, N_f = 7, N_b = 5, N_0 = 5)

INFO:__main__:The circuit used in the study:


The circuit is saved in ./results/qho/2025-08-29_01-59-16-214473


In [ ]:
# For the moment, while using emulators, there's no cost related
print(f"Theorical cost: {tracker.simulator_tasks_cost()}, with approximately {model.quantum_layer.shots_done} shots")

In [ ]:
# Loss history plot 
plt.semilogy(range(len(model.loss_history)), model.loss_history)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss Over Epochs")
plt.grid()

file_path = os.path.join(model.log_path, "loss_history.pdf")
plt.savefig(file_path, bbox_inches="tight")
plt.show()

plt.close(
    "all",
)

## Pruebas de resultados obtenidos

Ahora podemos considerar los resultados obtenidos para validar el comportamiento del modelo obtenido

In [27]:
# settings
number_of_points = 10

with torch.no_grad():
    # mesh of t - x evaluation
    # - Create mesh grid with float32
    if example.lower() == 'ho':
        t = np.linspace(0, T, number_of_points, dtype=np.float32)[:, None]
        x = np.linspace(-L, L, number_of_points, dtype=np.float32)[:, None]
    else:
        t = np.linspace(0, T, number_of_points, dtype=np.float32)[:, None]
        x = torch.linspace(0.0, L, number_of_points, device=DEVICE, dtype=DTYPE).unsqueeze(1)

    t, x = np.meshgrid(t, x);

    t = t.flatten()[:, None]
    x = x.flatten()[:, None]

    # - Generation of torch elements
    t_eval = torch.from_numpy(t); x_eval = torch.from_numpy(x)

    # ALR: Non-Classical evaluation
    psi_pred = model(torch.cat((t_eval, x_eval), dim=1))
    psi_r_pred = psi_pred[:, 0:1]
    psi_i_pred = psi_pred[:, 1:2]
    mod2_pred = (psi_r_pred**2 + psi_i_pred**2).squeeze(1).cpu().numpy()

    psi_r_true, psi_i_true, En = exact_eigenstate(n_level, t_eval, x_eval, L=L, mass=mass, hbar=hbar, omega=omega, example=example)
    mod2_true = (psi_r_true**2 + psi_i_true**2).squeeze(1).cpu().numpy()

# - Generation of input argument
X = (
    torch.hstack(
        (torch.from_numpy(t.flatten()[:, None]), torch.from_numpy(x.flatten()[:, None]))
    )
    .to(DEVICE)
    .to(torch.float32)
).cpu().detach().numpy()

In [ ]:
# Routine plots:
plt_prediction(
    logger,
    X,
    psi_r_true.cpu().detach().numpy(),
    psi_r_pred.cpu().detach().numpy(),
    psi_i_true.cpu().detach().numpy(),
    psi_i_pred.cpu().detach().numpy(),
)

In [ ]:
# Gráfico 1: |psi|^2 en t=T (PINN vs exacto)
plt.figure()
plt.plot(x_eval.squeeze(1).cpu().numpy(), mod2_true, label="|ψ|^2 exacto")
plt.plot(x_eval.squeeze(1).cpu().numpy(), mod2_pred, "--", label="|ψ|^2 PINN")
plt.title(f"|ψ(x,T)|^2 en pozo infinito (n={n_level})")
plt.xlabel("x")
plt.ylabel("|ψ|^2")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Gráfico 2: error absoluto en |psi|^2
plt.figure()
abs_err = np.abs(mod2_pred - mod2_true)
plt.plot(x_eval.squeeze(1).cpu().numpy(), abs_err, label="Error absoluto")
plt.title("Error absoluto en |ψ(x,T)|^2")
plt.xlabel("x")
plt.ylabel("Error")
plt.legend()
plt.grid()
plt.show()